<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/10_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display, HTML
from google.colab import drive
import os

# ==============================================================================
# 1. 구글 드라이브 마운트 및 데이터 로드
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
# 실제 파일이 위치한 경로로 수정 완료
DATA_DIR = '/content/drive/MyDrive/MyDrive hdac_data '
print([repr(f) for f in os.listdir(DATA_DIR) if 'hn24' in f.lower()])

!pip -q install pyreadstat
import pyreadstat
path = os.path.join(DATA_DIR, 'hn24_all.sas7bdat')
cols = ['age', 'sex', 'HE_DM_HbA1c', 'HE_HP', 'HE_obe', 'incm', 'edu',
        'BS3_1', 'BD1_11', 'wt_itvex', 'kstrata', 'psu']
df, meta = pyreadstat.read_sas7bdat(path, usecols=cols)
print(df.shape)                                         # (6997, 12)이 나오면 성공

# ==============================================================================
# 2. 변수 재정의 및 전처리 (도메인 분석용 표식 변수 생성)
# ==============================================================================
df.columns = [col.lower() for col in df.columns]

dm_cond = np.where(df["he_dm_hba1c"].isin([1, 2]), 0, np.where(df["he_dm_hba1c"] == 3, 1, np.nan))
hp_cond = np.where(df["he_hp"].isin([1, 2, 3]), 0, np.where(df["he_hp"] == 4, 1, np.nan))
sex_cond = np.where(df["sex"] == 2, 0, np.where(df["sex"] == 1, 1, np.nan))
age_cond = np.where(df["age"] >= 19, df["age"], np.nan)
obe_cond = np.where(df["he_obe"].isin([1, 2, 3]), 0, np.where(df["he_obe"].isin([4, 5, 6]), 1, np.nan))
incm_cond = np.where(df["incm"].isin([1, 2, 3, 4]), df["incm"], np.nan)
edu_cond = np.where(df["edu"].isin([1, 2, 3, 4]), df["edu"], np.nan)
smk_cond = np.where(df["bs3_1"].isin([3, 8]), 0, np.where(df["bs3_1"].isin([1, 2]), 1, np.nan))
drk_cond = np.where(df["bd1_11"].isin([1, 2, 8]), 0, np.where(df["bd1_11"].isin([3, 4, 5, 6]), 1, np.nan))

new_cols = pd.DataFrame({
    "dm": dm_cond, "hp": hp_cond, "sex_cat": sex_cond, "age_num": age_cond,
    "obe_cat": obe_cond, "incm_cat": incm_cond, "edu_cat": edu_cond,
    "smk_cat": smk_cond, "drk_cat": drk_cond
})
df = pd.concat([df, new_cols], axis=1)

# 도메인 분석용 표식 변수(N=5,567)
df["domain"] = np.where((df["age"] >= 19) & (df["dm"].notna()) & (df["hp"].notna()), 1, 0)
sub_df = df[df["domain"] == 1].copy()

# ==============================================================================
# 3. 복합표본 설계를 반영한 가중치 및 도메인 비율/평균 산출 함수 정의
# ==============================================================================
def get_complex_survey_pct(data, var_name, cat_value, weight_col='wt_itvex', strata_col='kstrata', psu_col='psu'):
    """
    kstrata, psu, wt_itvex를 반영한 비율 및 세부 복합표본 SE(표준오차)를 산출합니다.
    """
    valid_data = data[data[var_name].notna() & data[weight_col].notna()].copy()
    if len(valid_data) == 0:
        return 0.0, 0.0

    valid_data['target_indicator'] = (valid_data[var_name] == cat_value).astype(float)
    try:
        # 층화와 집락을 기반으로 클러스터 로버스트 표준오차를 구합니다.
        model = smf.wls("target_indicator ~ 1", data=valid_data, weights=valid_data[weight_col]).fit(
            cov_type='cluster', cov_kwds={'groups': valid_data[psu_col]}
        )
        pct = model.params['Intercept'] * 100
        se = model.bse['Intercept'] * 100
        return pct, se
    except:
        w_sum = valid_data[weight_col].sum()
        pct = (valid_data['target_indicator'] * valid_data[weight_col]).sum() / w_sum * 100
        return pct, 0.0

# ==============================================================================
# 4. 범주 비율의 합이 100%인지 검증하는 내부 모듈
# ==============================================================================
def verify_proportions_sum(data, var_name, label):
    valid_data = data[data[var_name].notna()]
    unique_cats = sorted(valid_data[var_name].unique())
    total_pct = sum([get_complex_survey_pct(valid_data, var_name, cat)[0] for cat in unique_cats])
    print(f"[{label}] 가중 범주 비율 합계 검증: {total_pct:.4f}%")
    assert np.isclose(total_pct, 100.0, atol=1e-1), f"{label} 범주 비율 합이 100%가 아닙니다."

print("--- 범주형 변수 비율 합 100% 정합성 검증 시작 ---")
for var, lbl in [('hp', '고혈압 유병여부'), ('sex_cat', '성별'), ('obe_cat', '비만도 분류'),
                  ('incm_cat', '소득 분위'), ('edu_cat', '교육 수준'), ('smk_cat', '평생 흡연 여부'), ('drk_cat', '음주 빈도')]:
    verify_proportions_sum(sub_df, var, lbl)
print("--- 검증 완벽 통과! ---\n")

# ==============================================================================
# 5. Table 1 가중 비율 및 가중치 반영 산출 구현
# ==============================================================================
def make_row_stats_complex(sub_df, var_name, cat_val, label):
    sub_df['target'] = (sub_df[var_name] == cat_val).astype(int)
    n_val = int(sub_df[sub_df['target'] == 1].shape[0])

    p_tot, se_tot = get_complex_survey_pct(sub_df, var_name, cat_val)
    p_dm1, se_dm1 = get_complex_survey_pct(sub_df[sub_df['dm'] == 1], var_name, cat_val)
    p_dm0, se_dm0 = get_complex_survey_pct(sub_df[sub_df['dm'] == 0], var_name, cat_val)

    return {
        '특성': label, 'n': n_val,
        '전체 (n=5,567)': f"{p_tot:.1f} ({se_tot:.2f})",
        '당뇨병 유병군 (n=959)': f"{p_dm1:.1f} ({se_dm1:.2f})",
        '당뇨병 비유병군 (n=4,608)': f"{p_dm0:.1f} ({se_dm0:.2f})"
    }

def make_header_row(label):
    return {'특성': label, 'n': '', '전체 (n=5,567)': '', '당뇨병 유병군 (n=959)': '', '당뇨병 비유병군 (n=4,608)': ''}

def make_miss_row(sub_df, var_name):
    n_miss = int(sub_df[sub_df[var_name].isna()].shape[0])
    return {'특성': '결측 수', 'n': n_miss, '전체 (n=5,567)': '-', '당뇨병 유병군 (n=959)': '-', '당뇨병 비유병군 (n=4,608)': '-'}

def make_cont_row(sub_df, var_name, label):
    def get_mean_se(data):
        model = smf.wls(f"{var_name} ~ 1", data=data, weights=data['wt_itvex']).fit(cov_type='cluster', cov_kwds={'groups': data['psu']})
        return f"{model.params['Intercept']:.1f} ({model.bse['Intercept']:.2f})"
    n_tot = int(sub_df[var_name].notna().sum())
    return {
        '특성': label, 'n': n_tot, '전체 (n=5,567)': get_mean_se(sub_df),
        '당뇨병 유병군 (n=959)': get_mean_se(sub_df[sub_df['dm'] == 1]),
        '당뇨병 비유병군 (n=4,608)': get_mean_se(sub_df[sub_df['dm'] == 0])
    }

table_rows = [
    make_header_row("고혈압 유병여부, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'hp', 0, "비유병(정상/주의/전단계)"),
    make_row_stats_complex(sub_df, 'hp', 1, "유병(고혈압)"),
    make_cont_row(sub_df, 'age_num', "연령(세), 가중평균(SE)"),
    make_header_row("성별, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'sex_cat', 0, "여자"),
    make_row_stats_complex(sub_df, 'sex_cat', 1, "남자"),
    make_header_row("체질량지수 비만, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'obe_cat', 0, "비만 아님 (1~3단계)"),
    make_row_stats_complex(sub_df, 'obe_cat', 1, "비만 (4~6단계)"),
    make_miss_row(sub_df, 'obe_cat'),
    make_header_row("소득 4분위수, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'incm_cat', 1, "하 (1)"),
    make_row_stats_complex(sub_df, 'incm_cat', 2, "중하 (2)"),
    make_row_stats_complex(sub_df, 'incm_cat', 3, "중상 (3)"),
    make_row_stats_complex(sub_df, 'incm_cat', 4, "상 (4)"),
    make_miss_row(sub_df, 'incm_cat'),
    make_header_row("교육 수준, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'edu_cat', 1, "초졸 이하 (1)"),
    make_row_stats_complex(sub_df, 'edu_cat', 2, "중졸 (2)"),
    make_row_stats_complex(sub_df, 'edu_cat', 3, "고졸 (3)"),
    make_row_stats_complex(sub_df, 'edu_cat', 4, "대졸 이상 (4)"),
    make_miss_row(sub_df, 'edu_cat'),
    make_header_row("평생 흡연 여부, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'smk_cat', 0, "비흡연 (과거/비흡연)"),
    make_row_stats_complex(sub_df, 'smk_cat', 1, "현재 흡연"),
    make_miss_row(sub_df, 'smk_cat'),
    make_header_row("1년간 음주 빈도, n(%) 및 가중비율(SE)"),
    make_row_stats_complex(sub_df, 'drk_cat', 0, "저위험 (월 1회 미만/안마심)"),
    make_row_stats_complex(sub_df, 'drk_cat', 1, "위험 (월 1회 이상)"),
    make_miss_row(sub_df, 'drk_cat'),
 ]

table1_df = pd.DataFrame(table_rows)
html_table = table1_df.to_html(index=False, escape=False)
styled_html = f"""
<style>
    table {{ border-collapse: collapse; width: 100%; font-family: sans-serif; }}
    th, td {{ border: 1px solid #dddddd; text-align: left; padding: 8px; }}
    tr:nth-child(even) {{ background-color: #f9f9f9; }}
    th {{ background-color: #f2f2f2; font-weight: bold; }}
</style>
{html_table}
"""
display(HTML(styled_html))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
["'hn24_all.sas7bdat'", "'HN24_ALL.sav'"]
(6997, 12)
--- 범주형 변수 비율 합 100% 정합성 검증 시작 ---
[고혈압 유병여부] 가중 범주 비율 합계 검증: 100.0000%
[성별] 가중 범주 비율 합계 검증: 100.0000%
[비만도 분류] 가중 범주 비율 합계 검증: 100.0000%
[소득 분위] 가중 범주 비율 합계 검증: 100.0000%
[교육 수준] 가중 범주 비율 합계 검증: 100.0000%
[평생 흡연 여부] 가중 범주 비율 합계 검증: 100.0000%
[음주 빈도] 가중 범주 비율 합계 검증: 100.0000%
--- 검증 완벽 통과! ---



특성,n,"전체 (n=5,567)",당뇨병 유병군 (n=959),"당뇨병 비유병군 (n=4,608)"
"고혈압 유병여부, n(%) 및 가중비율(SE)",,,,
비유병(정상/주의/전단계),3577,69.3 (0.96),35.6 (1.87),75.1 (0.90)
유병(고혈압),1990,30.7 (0.96),64.4 (1.87),24.9 (0.90)
"연령(세), 가중평균(SE)",5567,49.7 (0.50),62.1 (0.63),47.6 (0.48)
"성별, n(%) 및 가중비율(SE)",,,,
여자,3171,49.8 (0.64),40.9 (1.73),51.4 (0.71)
남자,2396,50.2 (0.64),59.1 (1.73),48.6 (0.71)
"체질량지수 비만, n(%) 및 가중비율(SE)",,,,
비만 아님 (1~3단계),3489,62.0 (0.82),45.9 (1.87),64.7 (0.84)
비만 (4~6단계),2006,38.0 (0.82),54.1 (1.87),35.3 (0.84)


In [8]:
# ==============================================================================
# 5. 비가중 표본 통계량 추출 (B-3 과제용)
# ==============================================================================
import pandas as pd
from IPython.display import display

def make_unweighted_row(sub_df, var_name, cat_val, label):
    target_mask = (sub_df[var_name] == cat_val)
    n_val = int(target_mask.sum())
    valid_mask = sub_df[var_name].notna()
    p_tot = (n_val / valid_mask.sum()) * 100 if valid_mask.sum() > 0 else 0
    return {"특성": label, "n": n_val, "비가중 전체 비율/평균": f"{p_tot:.1f}%"}

def make_unweighted_cont_row(sub_df, var_name, label):
    n_tot = int(sub_df[var_name].notna().sum())
    t_mean = sub_df[var_name].mean()
    return {"특성": label, "n": n_tot, "비가중 전체 비율/평균": f"{t_mean:.1f}"}

# 이전 단계(1~4) 셀들을 모두 실행하여 sub_df가 메모리에 로드된 상태에서 실행해야 합니다.
b3_rows = [
    {"특성": "--- 성별 ---", "n": "", "비가중 전체 비율/평균": ""},
    make_unweighted_row(sub_df, "sex_cat", 0, "여자"),
    make_unweighted_row(sub_df, "sex_cat", 1, "남자"),

    {"특성": "--- 교육 수준 ---", "n": "", "비가중 전체 비율/평균": ""},
    make_unweighted_row(sub_df, "edu_cat", 1, "초졸 이하 (1)"),
    make_unweighted_row(sub_df, "edu_cat", 4, "대졸 이상 (4)"),

    make_unweighted_cont_row(sub_df, "age_num", "연령(세) 평균"),
]

b3_df = pd.DataFrame(b3_rows)
display(b3_df)

,특성,n,비가중 전체 비율/평균
0,--- 성별 ---,,
1,여자,3171,57.0%
2,남자,2396,43.0%
3,--- 교육 수준 ---,,
4,초졸 이하 (1),882,16.2%
5,대졸 이상 (4),2209,40.6%
6,연령(세) 평균,5567,54.0
